In this notebook, we will be working through [Building makemore Part 4: Becoming a Backprop Ninja](https://www.youtube.com/watch?v=q8SA3rM6ckI&t=5204s) by Andrej Karpathy. This is the fifth video in the "Neural Networks: Zero to Hero" series.

The core idea of this lesson: up until now we've been calling `loss.backward()` and letting PyTorch compute all gradients for us via autograd. In this notebook, we **manually derive and implement every gradient** in the backward pass — from the loss all the way back to the embedding table. This forces us to understand exactly what autograd is doing under the hood, and builds the kind of low-level intuition that becomes critical when debugging training issues, implementing custom layers, or reading research papers that describe novel architectures.

Karpathy calls this "becoming a backprop ninja" — the goal is to be so comfortable with the chain rule and tensor shapes that you can backpropagate through any computation graph by hand.


In [1]:
# lets copy over boiler plate code from before

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np
%matplotlib inline

In [2]:
# lets copy over some stuff from the previous notebook

# utility to read dataset
DATASET_PATH = '../names.txt'
SPECIAL_TOKEN = "."
block_size = 3
g = torch.Generator().manual_seed(2147483647)

def get_dataset():
    with open(DATASET_PATH, 'r') as f:
        rows = [row.strip() for row in f.readlines()]
    return rows

# Load dataset
words = get_dataset()
print(f"{len(words)} names loaded")
print(f"Examples: {words[:8]}")

# Build character mappings — identical to lesson 2
# '.' is our special start/end token at index 0, then a=1, b=2, ..., z=26
all_characters = [SPECIAL_TOKEN] + sorted(list(set(''.join(words))))
stoi = {s: i for i, s in enumerate(all_characters)}
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print(f"Vocabulary size: {vocab_size}")
print(f"Mappings: {itos}")


32033 names loaded
Examples: ['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']
Vocabulary size: 27
Mappings: {0: '.', 1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z'}


In [3]:
def build_dataset(words, block_size=3):
    """
    Convert a list of words into (X, Y) tensors for training.

    X shape: (N, block_size) — each row is a context window of character indices
    Y shape: (N,) — each element is the target character index

    This function will be called three times: once each for train, val, and test splits.
    """
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + SPECIAL_TOKEN:
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.seed(42)
random.shuffle(words)

# 80/10/10 split at the word level
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1], block_size=3)     # training
Xdev, Ydev = build_dataset(words[n1:n2], block_size=3)  # validation ("dev" set)
Xte, Yte = build_dataset(words[n2:], block_size=3)      # test


print(f"Training:   {Xtr.shape[0]:>7,} examples from {n1:,} names")
print(f"Validation: {Xdev.shape[0]:>7,} examples from {n2-n1:,} names")
print(f"Test:       {Xte.shape[0]:>7,} examples from {len(words)-n2:,} names")

Training:   182,625 examples from 25,626 names
Validation:  22,655 examples from 3,203 names
Test:        22,866 examples from 3,204 names


# Section 1: Atomic Forward Pass and Manual Backpropagation

We are making a few changes to `get_params` compared to the previous lesson. Recall that when we have a batchnorm layer after a linear layer, the bias in the linear layer is redundant — batchnorm subtracts the mean anyway, so the bias gets cancelled out. In the previous notebook we removed it. Similarly, we had initialized `bngain` to ones and `bnbias` to zeros (their mathematically "neutral" values).

However, for this notebook we **intentionally keep the bias and initialize all parameters to small random non-zero values**. Why? Because we want every parameter to have a non-trivial gradient so we can verify our manual backprop against PyTorch's autograd. If a parameter were initialized to zero or effectively cancelled out, its gradient might be trivially zero, which wouldn't test our implementation.


In [4]:
def get_params(embed_dim, block_size, n_hidden):

    tanh_gain = 5./3  # correction factor for tanh squashing variance

    g = torch.Generator().manual_seed(2147483647)
    C  = torch.randn((vocab_size, embed_dim),            generator=g)
    W1 = torch.randn((embed_dim * block_size, n_hidden), generator=g) * (tanh_gain / (embed_dim * block_size) ** 0.5)
    b1 = torch.randn(n_hidden,                        generator=g) * 0.1
    W2 = torch.randn((n_hidden, vocab_size),           generator=g) * 0.1
    b2 = torch.randn(vocab_size,                       generator=g) * 0.1

    bngain = torch.randn((1, n_hidden))*0.1 + 1.0
    bnbias = torch.randn((1, n_hidden))*0.1

    # Ignore tracking for this excercise since these are not trainable and wont be part of backprop
    # bnstd_running = torch.ones((1, n_hidden))
    # bnmean_running = torch.zeros((1, n_hidden))

    parameters = [C, W1, b1, W2, b2, bngain, bnbias]
    for p in parameters:
        p.requires_grad = True

    return parameters

Now, at the end of one of previous lesson's [notebook](../lesson_4_makemore_batchnorm/makemore_initialization_batchnorm_part1.ipynb#solution-2-running-average-exponential-moving-average), we had seen this forward and backward pass (essentially before we pytorchyfied our code)

```python
ixds = torch.randint(0, Xtr.shape[0], (batch_size,))
mini_batch_inp, mini_batch_target = Xtr[ixds], Ytr[ixds]

# Forward pass
emb = C[mini_batch_inp]                                        # (32, 3, 10)
emb_cat = emb.view(emb.shape[0], -1)                            # (32, 30)
hidden_layer_preactivation = emb_cat @ W1 + b1                  # (32, 200)

# CONVERT hidden_layer_preactivation to unit gaussian
_mean = hidden_layer_preactivation.mean(axis=0, keepdim=True) # (1, 200) take mean across the samples in the mini batch
_std = hidden_layer_preactivation.std(axis=0, keepdim=True) # (1, 200) take std across the samples in the mini batch

hidden_layer_preactivation = (hidden_layer_preactivation - _mean) / _std # convert to unit gaussian

# move the running average slightly based on the currnet mean and std directions
with torch.no_grad():
    bnmean_running = 0.999 * bnmean_running + 0.001 * _mean
    bnstd_running = 0.999 * bnstd_running + 0.001 * _std

# scale and shift
hidden_layer_preactivation = hidden_layer_preactivation*bngain + bnbias

h = torch.tanh(hidden_layer_preactivation)  # (32, 200)
logits = h @ W2 + b2                                           # (32, 27)
loss = F.cross_entropy(logits, mini_batch_target)

# Backward pass
for p in parameters:
    p.grad = None
loss.backward()

# Learning rate step decay: 0.1 for first 100K steps, then 0.01
lr = 0.1 if i < 100000 else 0.01
for p in parameters:
    p.data += -lr * p.grad
```

However, for this initial manual backprop exercise, we will first break down each step of the forward pass such that it only constitutes atomic arithmetic operations. This will allow us to backprop through the entire model step by step.


In [5]:
# utility to compare tensors.

def cmp(s, dt, t):
    """
    We will use this to compare our calculation of gradient (dt) to pytorch's calculations (t)
    """

    exact = torch.all(dt == t.grad).item()
    approx = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()

    print(f'{s:15s} | exact: {str(exact):5s} | approximate: {str(approx):5s} | maxdiff: {maxdiff}')

In [6]:
# Lets get our parameters first, and create a single forward pass to step through

embed_dim = 10
n_hidden = 64

n = batch_size = 32
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

C, W1, b1, W2, b2, bngain, bnbias = get_params(embed_dim, block_size, n_hidden)

max_name = 12   # length of the longest name ("bngain.shape" / "bnbias.shape")

# for x in [Xb, Yb, C, W1, b1, W2, b2, bngain, bnbias]:
print(f"{'Xb.shape':<{max_name}} = {Xb.shape!s:>25}")
print(f"{'Yb.shape':<{max_name}} = {Yb.shape!s:>25}")
print(f"{'C.shape':<{max_name}} = {C.shape!s:>25}")
print(f"{'W1.shape':<{max_name}} = {W1.shape!s:>25}")
print(f"{'b1.shape':<{max_name}} = {b1.shape!s:>25}")
print(f"{'W2.shape':<{max_name}} = {W2.shape!s:>25}")
print(f"{'b2.shape':<{max_name}} = {b2.shape!s:>25}")
print(f"{'bngain.shape':<{max_name}} = {bngain.shape!s:>25}")
print(f"{'bnbias.shape':<{max_name}} = {bnbias.shape!s:>25}")

Xb.shape     =       torch.Size([32, 3])
Yb.shape     =          torch.Size([32])
C.shape      =      torch.Size([27, 10])
W1.shape     =      torch.Size([30, 64])
b1.shape     =          torch.Size([64])
W2.shape     =      torch.Size([64, 27])
b2.shape     =          torch.Size([27])
bngain.shape =       torch.Size([1, 64])
bnbias.shape =       torch.Size([1, 64])


In [7]:
parameters = [C, W1, b1, W2, b2, bngain, bnbias]

### The Atomic Forward Pass

Below we break the forward pass into the **smallest possible steps** — each line is a single arithmetic operation on tensors. In the previous notebook we wrote things like `F.cross_entropy(logits, targets)`, which internally does subtract-max, exponentiate, sum, divide, log, and index — all in one call. Here we do each of those steps explicitly so we have a named intermediate tensor for every node in the computation graph.

**Variable naming convention** (matching Karpathy's video):

- **`emb` / `embcat`**: embedding lookup and flattened (concatenated) embedding
- **`hprebn`**: hidden layer pre-batchnorm (output of linear layer 1)
- **`bnmeani`, `bndiff`, `bndiff2`, `bnvar`, `bnvar_inv`, `bnraw`**: the individual steps of batch normalization — mean, centered values, squared differences, variance, inverse std, and the normalized output
- **`hpreact`**: hidden layer pre-activation (after batchnorm scale+shift, before tanh)
- **`h`**: hidden activations (after tanh)
- **`logits` → `logit_maxes` → `norm_logits` → `counts` → `counts_sum` → `counts_sum_inv` → `probs` → `logprobs`**: the individual steps of cross-entropy loss (numeric stabilization, softmax via exp/sum/divide, then log)

Every one of these intermediates will get a gradient (`d`-prefixed version) in the manual backprop exercise.


In [8]:
# atomic forward pass

# --- Embedding layer ---
emb = C[Xb]                                                     # (32, 3, 10) lookup each context char's embedding vector
embcat = emb.view(n, -1)                                         # (32, 30)    flatten 3 embeddings into one row

# --- Linear layer 1 ---
hprebn = embcat @ W1 + b1                                        # (32, 64)    pre-batchnorm hidden activations

# --- Batch normalization (broken into atomic steps) ---
bnmeani = (1./n) * hprebn.sum(axis=0, keepdim=True)              # (1, 64)     mean of each neuron across the batch
bndiff = hprebn - bnmeani                                        # (32, 64)    center: subtract mean
bndiff2 = bndiff ** 2                                            # (32, 64)    squared deviations

bnvar = 1./(n-1) * (bndiff2).sum(axis=0, keepdim=True)          # (1, 64)     variance (Bessel-corrected, / n-1)
bnvar_inv = (bnvar + 1e-5)**-0.5                                 # (1, 64)     inverse std (with epsilon for numerical stability)

bnraw = bndiff * bnvar_inv                                       # (32, 64)    normalized activations (zero mean, unit variance)
hpreact = bngain * bnraw + bnbias                                # (32, 64)    scale and shift (learnable affine transform)

# --- Non-linearity ---
h = torch.tanh(hpreact)                                          # (32, 64)    tanh activation

# --- Linear layer 2 (output) ---
logits = h @ W2 + b2                                             # (32, 27)    raw scores over vocab

# --- Cross-entropy loss (broken into atomic steps) ---
logit_maxes = logits.max(axis=1, keepdim=True).values            # (32, 1)     max per sample for numeric stability
norm_logits = logits - logit_maxes                                # (32, 27)    shift logits (doesn't change softmax output)

counts = norm_logits.exp()                                       # (32, 27)    exponentiate (unnormalized probabilities)
counts_sum = counts.sum(axis=1, keepdim=True)                    # (32, 1)     partition function Z per sample
counts_sum_inv = counts_sum ** -1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...

probs = counts * counts_sum_inv                                  # (32, 27)    softmax probabilities
logprobs = probs.log()                                           # (32, 27)    log probabilities

loss = (-1./n) * logprobs[range(n), Yb].sum()                   # scalar      negative log-likelihood (mean over batch)

# --- PyTorch backward pass (for verification) ---
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss


tensor(3.4705, grad_fn=<MulBackward0>)

### Why `retain_grad()` and `cmp()`?

By default, PyTorch only stores `.grad` for leaf tensors (parameters). All the intermediates like `bnraw`, `probs`, etc. are non-leaf — their gradients get computed during `loss.backward()` but are immediately discarded. Calling `t.retain_grad()` tells PyTorch to keep them, so we can compare our hand-derived gradients against PyTorch's ground truth using `cmp()`.

The exercise below asks you to compute `dlogprobs`, `dprobs`, ..., `dC` — the gradient of the loss with respect to each intermediate, working backwards through the computation graph one step at a time.


In [9]:
# Exercise 1: backprop through the whole thing manually,
# backpropagating through exactly all of the variables
# as they are defined in the forward pass above, one by one

# -----------------
# YOUR CODE HERE :)
# -----------------

# cmp('logprobs', dlogprobs, logprobs)
# cmp('probs', dprobs, probs)
# cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
# cmp('counts_sum', dcounts_sum, counts_sum)
# cmp('counts', dcounts, counts)
# cmp('norm_logits', dnorm_logits, norm_logits)
# cmp('logit_maxes', dlogit_maxes, logit_maxes)
# cmp('logits', dlogits, logits)
# cmp('h', dh, h)
# cmp('W2', dW2, W2)
# cmp('b2', db2, b2)
# cmp('hpreact', dhpreact, hpreact)
# cmp('bngain', dbngain, bngain)
# cmp('bnbias', dbnbias, bnbias)
# cmp('bnraw', dbnraw, bnraw)
# cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
# cmp('bnvar', dbnvar, bnvar)
# cmp('bndiff2', dbndiff2, bndiff2)
# cmp('bndiff', dbndiff, bndiff)
# cmp('bnmeani', dbnmeani, bnmeani)
# cmp('hprebn', dhprebn, hprebn)
# cmp('embcat', dembcat, embcat)
# cmp('W1', dW1, W1)
# cmp('b1', db1, b1)
# cmp('emb', demb, emb)
# cmp('C', dC, C)

## Theory

Okay so we looked at the forward pass and we now want to get the derivative of the loss with respect to all the trainable variables from the forward pass. Before we do that, lets cover some theory that will help us derive the gradients.

### Chain Rule

The chain rule is the backbone of backpropagation. If the loss $L$ depends on some variable $x$ only through an intermediate variable $y$ — i.e. the computation graph looks like $\cdots \to x \to y \to \cdots \to L$ — then:

$$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \cdot \frac{\partial y}{\partial x}$$

In backprop, we work **backwards** through the graph. At each step we already have $\frac{\partial L}{\partial y}$ (the **upstream gradient**, computed in the previous backprop step). Our only job is to compute the **local gradient** $\frac{\partial y}{\partial x}$ and multiply. This is what makes backprop so elegant — each node only needs to know its own local derivative, and the chain rule stitches everything together.

When a variable feeds into **multiple** downstream paths, we sum the contributions (multivariate chain rule):

$$\frac{\partial L}{\partial x} = \sum_i \frac{\partial L}{\partial y_i} \cdot \frac{\partial y_i}{\partial x}$$

We'll see this repeatedly — for example, `bndiff` feeds into both `bndiff2` and `bnraw`, so its gradient is the sum of the gradients flowing back from both paths.


### Matrix Multiplication Derivatives (used in linear layers)

A linear layer computes $h = A @ B + C$, where $A$ is the input, $B$ is the weight matrix, and $C$ is the bias (broadcast). Given the upstream gradient $\frac{\partial L}{\partial h}$, we need to derive $\frac{\partial L}{\partial A}$, $\frac{\partial L}{\partial B}$, and $\frac{\partial L}{\partial C}$.

**Setup:** Let all matrices be 2×2, with the bias $C$ being 1×2 (broadcast across rows):

$$\begin{bmatrix} h_{11} & h_{12} \\ h_{21} & h_{22} \end{bmatrix} = \begin{bmatrix} a_{11} & a_{12} \\ a_{21} & a_{22} \end{bmatrix} @ \begin{bmatrix} b_{11} & b_{12} \\ b_{21} & b_{22} \end{bmatrix} + \begin{bmatrix} c_1 & c_2 \\ c_1 & c_2 \end{bmatrix}$$

Expanding the matrix multiply, each element of $h$ is:

$$h_{11} = a_{11} b_{11} + a_{12} b_{21} + c_1$$
$$h_{12} = a_{11} b_{12} + a_{12} b_{22} + c_2$$
$$h_{21} = a_{21} b_{11} + a_{22} b_{21} + c_1$$
$$h_{22} = a_{21} b_{12} + a_{22} b_{22} + c_2$$

#### Deriving $\frac{\partial L}{\partial A}$

By the chain rule, $\frac{\partial L}{\partial A} = \frac{\partial L}{\partial h} \cdot \frac{\partial h}{\partial A}$. We need to figure out which elements of $h$ depend on each element of $A$, then sum those contributions.

$a_{11}$ appears in $h_{11}$ and $h_{12}$ (the entire first row of $h$):

$$\frac{\partial L}{\partial a_{11}} = \frac{\partial L}{\partial h_{11}} \cdot \frac{\partial h_{11}}{\partial a_{11}} + \frac{\partial L}{\partial h_{12}} \cdot \frac{\partial h_{12}}{\partial a_{11}} = \frac{\partial L}{\partial h_{11}} \cdot b_{11} + \frac{\partial L}{\partial h_{12}} \cdot b_{12}$$

$a_{12}$ also appears in $h_{11}$ and $h_{12}$:

$$\frac{\partial L}{\partial a_{12}} = \frac{\partial L}{\partial h_{11}} \cdot \frac{\partial h_{11}}{\partial a_{12}} + \frac{\partial L}{\partial h_{12}} \cdot \frac{\partial h_{12}}{\partial a_{12}} = \frac{\partial L}{\partial h_{11}} \cdot b_{21} + \frac{\partial L}{\partial h_{12}} \cdot b_{22}$$

$a_{21}$ appears in $h_{21}$ and $h_{22}$ (the entire second row of $h$):

$$\frac{\partial L}{\partial a_{21}} = \frac{\partial L}{\partial h_{21}} \cdot \frac{\partial h_{21}}{\partial a_{21}} + \frac{\partial L}{\partial h_{22}} \cdot \frac{\partial h_{22}}{\partial a_{21}} = \frac{\partial L}{\partial h_{21}} \cdot b_{11} + \frac{\partial L}{\partial h_{22}} \cdot b_{12}$$

$a_{22}$ also appears in $h_{21}$ and $h_{22}$:

$$\frac{\partial L}{\partial a_{22}} = \frac{\partial L}{\partial h_{21}} \cdot \frac{\partial h_{21}}{\partial a_{22}} + \frac{\partial L}{\partial h_{22}} \cdot \frac{\partial h_{22}}{\partial a_{22}} = \frac{\partial L}{\partial h_{21}} \cdot b_{21} + \frac{\partial L}{\partial h_{22}} \cdot b_{22}$$

Assembling these into a matrix:

$$\frac{\partial L}{\partial A} = \begin{bmatrix} \frac{\partial L}{\partial h_{11}} \cdot b_{11} + \frac{\partial L}{\partial h_{12}} \cdot b_{12} & \frac{\partial L}{\partial h_{11}} \cdot b_{21} + \frac{\partial L}{\partial h_{12}} \cdot b_{22} \\ \frac{\partial L}{\partial h_{21}} \cdot b_{11} + \frac{\partial L}{\partial h_{22}} \cdot b_{12} & \frac{\partial L}{\partial h_{21}} \cdot b_{21} + \frac{\partial L}{\partial h_{22}} \cdot b_{22} \end{bmatrix} = \begin{bmatrix} \frac{\partial L}{\partial h_{11}} & \frac{\partial L}{\partial h_{12}} \\ \frac{\partial L}{\partial h_{21}} & \frac{\partial L}{\partial h_{22}} \end{bmatrix} @ \begin{bmatrix} b_{11} & b_{21} \\ b_{12} & b_{22} \end{bmatrix}$$

$$\boxed{\frac{\partial L}{\partial A} = \frac{\partial L}{\partial h} \; @ \; B^T}$$

#### Deriving $\frac{\partial L}{\partial B}$

Now we do the same for $B$. We need to find which elements of $h$ depend on each element of $B$.

$b_{11}$ appears in $h_{11}$ and $h_{21}$ (the entire first column of $h$):

$$\frac{\partial L}{\partial b_{11}} = \frac{\partial L}{\partial h_{11}} \cdot \frac{\partial h_{11}}{\partial b_{11}} + \frac{\partial L}{\partial h_{21}} \cdot \frac{\partial h_{21}}{\partial b_{11}} = \frac{\partial L}{\partial h_{11}} \cdot a_{11} + \frac{\partial L}{\partial h_{21}} \cdot a_{21}$$

$b_{12}$ appears in $h_{12}$ and $h_{22}$:

$$\frac{\partial L}{\partial b_{12}} = \frac{\partial L}{\partial h_{12}} \cdot \frac{\partial h_{12}}{\partial b_{12}} + \frac{\partial L}{\partial h_{22}} \cdot \frac{\partial h_{22}}{\partial b_{12}} = \frac{\partial L}{\partial h_{12}} \cdot a_{11} + \frac{\partial L}{\partial h_{22}} \cdot a_{21}$$

$b_{21}$ appears in $h_{11}$ and $h_{21}$:

$$\frac{\partial L}{\partial b_{21}} = \frac{\partial L}{\partial h_{11}} \cdot \frac{\partial h_{11}}{\partial b_{21}} + \frac{\partial L}{\partial h_{21}} \cdot \frac{\partial h_{21}}{\partial b_{21}} = \frac{\partial L}{\partial h_{11}} \cdot a_{12} + \frac{\partial L}{\partial h_{21}} \cdot a_{22}$$

$b_{22}$ appears in $h_{12}$ and $h_{22}$:

$$\frac{\partial L}{\partial b_{22}} = \frac{\partial L}{\partial h_{12}} \cdot \frac{\partial h_{12}}{\partial b_{22}} + \frac{\partial L}{\partial h_{22}} \cdot \frac{\partial h_{22}}{\partial b_{22}} = \frac{\partial L}{\partial h_{12}} \cdot a_{12} + \frac{\partial L}{\partial h_{22}} \cdot a_{22}$$

Assembling into a matrix:

$$\frac{\partial L}{\partial B} = \begin{bmatrix} a_{11} \cdot \frac{\partial L}{\partial h_{11}} + a_{21} \cdot \frac{\partial L}{\partial h_{21}} & a_{11} \cdot \frac{\partial L}{\partial h_{12}} + a_{21} \cdot \frac{\partial L}{\partial h_{22}} \\ a_{12} \cdot \frac{\partial L}{\partial h_{11}} + a_{22} \cdot \frac{\partial L}{\partial h_{21}} & a_{12} \cdot \frac{\partial L}{\partial h_{12}} + a_{22} \cdot \frac{\partial L}{\partial h_{22}} \end{bmatrix} = \begin{bmatrix} a_{11} & a_{21} \\ a_{12} & a_{22} \end{bmatrix} @ \begin{bmatrix} \frac{\partial L}{\partial h_{11}} & \frac{\partial L}{\partial h_{12}} \\ \frac{\partial L}{\partial h_{21}} & \frac{\partial L}{\partial h_{22}} \end{bmatrix}$$

$$\boxed{\frac{\partial L}{\partial B} = A^T \; @ \; \frac{\partial L}{\partial h}}$$

#### Deriving $\frac{\partial L}{\partial C}$

The bias $C$ has shape 1×2 but was broadcast to 2×2 in the forward pass — $C$ was copied across the rows (axis=0) so that each sample gets the same bias added. This means each $c_j$ influenced multiple elements of $h$, so in the backward pass we need to **sum the gradients along the same axis that was broadcast** (axis=0). This is a general rule: whenever a tensor is broadcast along some axis in the forward pass, the backward pass sums the upstream gradient along that same axis.

Concretely, $c_1$ appears in $h_{11}$ and $h_{21}$, and $c_2$ appears in $h_{12}$ and $h_{22}$:

$$\frac{\partial L}{\partial c_1} = \frac{\partial L}{\partial h_{11}} \cdot \frac{\partial h_{11}}{\partial c_1} + \frac{\partial L}{\partial h_{21}} \cdot \frac{\partial h_{21}}{\partial c_1} = \frac{\partial L}{\partial h_{11}} \cdot 1 + \frac{\partial L}{\partial h_{21}} \cdot 1$$

$$\frac{\partial L}{\partial c_2} = \frac{\partial L}{\partial h_{12}} \cdot \frac{\partial h_{12}}{\partial c_2} + \frac{\partial L}{\partial h_{22}} \cdot \frac{\partial h_{22}}{\partial c_2} = \frac{\partial L}{\partial h_{12}} \cdot 1 + \frac{\partial L}{\partial h_{22}} \cdot 1$$

This is just summing each column of $\frac{\partial L}{\partial h}$ — i.e. summing along axis=0, exactly the axis $C$ was broadcast along:

$$\boxed{\frac{\partial L}{\partial C} = \frac{\partial L}{\partial h}\texttt{.sum(axis=0)}}$$

#### Summary for $h = A \; @ \; B + C$

| Gradient                        | Formula                                              | Intuition                                                                         |
| ------------------------------- | ---------------------------------------------------- | --------------------------------------------------------------------------------- |
| $\frac{\partial L}{\partial A}$ | $\frac{\partial L}{\partial h} \; @ \; B^T$          | Each row of $A$ contributed to the corresponding row of $h$ via $B$               |
| $\frac{\partial L}{\partial B}$ | $A^T \; @ \; \frac{\partial L}{\partial h}$          | Each column of $B$ contributed to the corresponding column of $h$ via $A$         |
| $\frac{\partial L}{\partial C}$ | $\frac{\partial L}{\partial h}\texttt{.sum(axis=0)}$ | Bias was broadcast (copied across rows), so we sum the gradient back along axis=0 |


### Element-wise Operations with Broadcasting

Many operations in our forward pass are element-wise multiplications where one operand is broadcast. Let's derive the gradients for $A = B * C$ where $B$ is 2×2 and $C$ is 1×2 (broadcast across rows).

**Quick scalar intuition:** For a simple scalar product $a = b \cdot c$, the derivatives are $\frac{\partial a}{\partial b} = c$ and $\frac{\partial a}{\partial c} = b$. The tensor case is the same idea, just applied element-wise with broadcasting.

**Setup:** $B$ is 2×2, $C$ is 1×2 (broadcast along axis=0 to match $B$):

$$\begin{bmatrix} a_{11} & a_{12} \\ a_{21} & a_{22} \end{bmatrix} = \begin{bmatrix} b_{11} & b_{12} \\ b_{21} & b_{22} \end{bmatrix} * \begin{bmatrix} c_1 & c_2 \\ c_1 & c_2 \end{bmatrix}$$

Expanding element-wise:

$$a_{11} = b_{11} \cdot c_1$$
$$a_{12} = b_{12} \cdot c_2$$
$$a_{21} = b_{21} \cdot c_1$$
$$a_{22} = b_{22} \cdot c_2$$

Given $\frac{\partial L}{\partial A}$, we want to calculate $\frac{\partial L}{\partial B}$ and $\frac{\partial L}{\partial C}$.

#### Deriving $\frac{\partial L}{\partial B}$

Each $b_{ij}$ only affects the corresponding $a_{ij}$ (element-wise ops are independent per-element), so:

$$\frac{\partial L}{\partial b_{11}} = \frac{\partial L}{\partial a_{11}} \cdot \frac{\partial a_{11}}{\partial b_{11}} = \frac{\partial L}{\partial a_{11}} \cdot c_1$$

$$\frac{\partial L}{\partial b_{12}} = \frac{\partial L}{\partial a_{12}} \cdot \frac{\partial a_{12}}{\partial b_{12}} = \frac{\partial L}{\partial a_{12}} \cdot c_2$$

$$\frac{\partial L}{\partial b_{21}} = \frac{\partial L}{\partial a_{21}} \cdot \frac{\partial a_{21}}{\partial b_{21}} = \frac{\partial L}{\partial a_{21}} \cdot c_1$$

$$\frac{\partial L}{\partial b_{22}} = \frac{\partial L}{\partial a_{22}} \cdot \frac{\partial a_{22}}{\partial b_{22}} = \frac{\partial L}{\partial a_{22}} \cdot c_2$$

Assembling into a matrix:

$$\frac{\partial L}{\partial B} = \begin{bmatrix} \frac{\partial L}{\partial a_{11}} \cdot c_1 & \frac{\partial L}{\partial a_{12}} \cdot c_2 \\ \frac{\partial L}{\partial a_{21}} \cdot c_1 & \frac{\partial L}{\partial a_{22}} \cdot c_2 \end{bmatrix} = \frac{\partial L}{\partial A} * \begin{bmatrix} c_1 & c_2 \\ c_1 & c_2 \end{bmatrix}$$

The local gradient is $C$ broadcast along axis=0 to match $B$'s shape — which is the same broadcasting that happened in the forward pass:

$$\boxed{\frac{\partial L}{\partial B} = \frac{\partial L}{\partial A} * C \quad \text{(with } C \text{ broadcast along axis=0)}}$$

**Intuition for $\frac{\partial L}{\partial B}$:** $B$ has the same shape as the output $A$, so no broadcasting happened for $B$ — there is a clean one-to-one mapping between each $b_{ij}$ and its corresponding $a_{ij}$. This means the gradient just flows straight back element-wise: each element of $B$ gets the upstream gradient at that position, scaled by the local derivative (the corresponding value of $C$). The gradient for $B$ has the same shape as $B$ with no summing needed.

#### Deriving $\frac{\partial L}{\partial C}$

Since $C$ was broadcast along axis=0, each $c_j$ influenced multiple elements of $A$. Specifically, $c_1$ appears in $a_{11}$ and $a_{21}$, and $c_2$ appears in $a_{12}$ and $a_{22}$:

$$\frac{\partial L}{\partial c_1} = \frac{\partial L}{\partial a_{11}} \cdot \frac{\partial a_{11}}{\partial c_1} + \frac{\partial L}{\partial a_{21}} \cdot \frac{\partial a_{21}}{\partial c_1} = \frac{\partial L}{\partial a_{11}} \cdot b_{11} + \frac{\partial L}{\partial a_{21}} \cdot b_{21}$$

$$\frac{\partial L}{\partial c_2} = \frac{\partial L}{\partial a_{12}} \cdot \frac{\partial a_{12}}{\partial c_2} + \frac{\partial L}{\partial a_{22}} \cdot \frac{\partial a_{22}}{\partial c_2} = \frac{\partial L}{\partial a_{12}} \cdot b_{12} + \frac{\partial L}{\partial a_{22}} \cdot b_{22}$$

Assembling into a row vector:

$$\frac{\partial L}{\partial C} = \begin{bmatrix} \frac{\partial L}{\partial a_{11}} \cdot b_{11} + \frac{\partial L}{\partial a_{21}} \cdot b_{21} & \frac{\partial L}{\partial a_{12}} \cdot b_{12} + \frac{\partial L}{\partial a_{22}} \cdot b_{22} \end{bmatrix}$$

This is first doing the element-wise multiply $\frac{\partial L}{\partial A} * B$ (which gives us a 2×2 matrix), and then summing along axis=0 (the axis that was broadcast):

$$\boxed{\frac{\partial L}{\partial C} = \left(\frac{\partial L}{\partial A} * B\right)\texttt{.sum(axis=0)}}$$

**Intuition for $\frac{\partial L}{\partial C}$:** Broadcasting is secretly _copying_ — in the forward pass, $c_1$ was copied to every row so it could participate in multiple multiplications (with $b_{11}$ in row 1, with $b_{21}$ in row 2, etc.). Each of those copies produced a separate output element, and each output element has its own upstream gradient. In the backward pass, the gradient for $c_1$ must account for _all the places $c_1$ was used_, so we collect (sum) the gradient contributions from every row. This is why broadcasting forward along an axis always means summing backward along that same axis — we're "un-broadcasting" by accumulating the gradients from all the copies.

Notice the asymmetry: $B$'s gradient has the same shape as $B$ (no summing needed — one-to-one mapping), while $C$'s gradient requires summing to collapse back to $C$'s original smaller shape (many-to-one mapping from outputs back to the broadcast source).

#### Summary for $A = B * C$ (element-wise with broadcast)

| Gradient                        | Formula                                                               | Intuition                                                                       |
| ------------------------------- | --------------------------------------------------------------------- | ------------------------------------------------------------------------------- |
| $\frac{\partial L}{\partial B}$ | $\frac{\partial L}{\partial A} * C$                                   | No broadcast for $B$ — gradient flows straight back element-wise, scaled by $C$ |
| $\frac{\partial L}{\partial C}$ | $\left(\frac{\partial L}{\partial A} * B\right)\texttt{.sum(axis=0)}$ | $C$ was broadcast (copied) along axis=0, so we sum gradient back along axis=0   |


#### What if $C$ is broadcast along columns (axis=1) instead?

In the example above, $C$ had shape 1×2 and was broadcast along axis=0 (copied across rows). But we can also have the opposite — $C$ has shape 2×1 and is broadcast along axis=1 (copied across columns).

Let's derive this explicitly. Now $B$ is 2×2 and $C$ is 2×1 (broadcast along axis=1):

$$\begin{bmatrix} a_{11} & a_{12} \\ a_{21} & a_{22} \end{bmatrix} = \begin{bmatrix} b_{11} & b_{12} \\ b_{21} & b_{22} \end{bmatrix} * \begin{bmatrix} c_1 & c_1 \\ c_2 & c_2 \end{bmatrix}$$

Expanding element-wise:

$$a_{11} = b_{11} \cdot c_1$$
$$a_{12} = b_{12} \cdot c_1$$
$$a_{21} = b_{21} \cdot c_2$$
$$a_{22} = b_{22} \cdot c_2$$

**$\frac{\partial L}{\partial B}$** is the same story as before — one-to-one mapping, gradient flows straight back:

$$\frac{\partial L}{\partial b_{11}} = \frac{\partial L}{\partial a_{11}} \cdot c_1, \quad \frac{\partial L}{\partial b_{12}} = \frac{\partial L}{\partial a_{12}} \cdot c_1, \quad \frac{\partial L}{\partial b_{21}} = \frac{\partial L}{\partial a_{21}} \cdot c_2, \quad \frac{\partial L}{\partial b_{22}} = \frac{\partial L}{\partial a_{22}} \cdot c_2$$

$$\boxed{\frac{\partial L}{\partial B} = \frac{\partial L}{\partial A} * C \quad \text{(with } C \text{ broadcast along axis=1)}}$$

**$\frac{\partial L}{\partial C}$** — now each $c_i$ was copied across the _columns_ of its row. So $c_1$ appears in $a_{11}$ and $a_{12}$ (the entire first row), and $c_2$ appears in $a_{21}$ and $a_{22}$ (the entire second row):

$$\frac{\partial L}{\partial c_1} = \frac{\partial L}{\partial a_{11}} \cdot \frac{\partial a_{11}}{\partial c_1} + \frac{\partial L}{\partial a_{12}} \cdot \frac{\partial a_{12}}{\partial c_1} = \frac{\partial L}{\partial a_{11}} \cdot b_{11} + \frac{\partial L}{\partial a_{12}} \cdot b_{12}$$

$$\frac{\partial L}{\partial c_2} = \frac{\partial L}{\partial a_{21}} \cdot \frac{\partial a_{21}}{\partial c_2} + \frac{\partial L}{\partial a_{22}} \cdot \frac{\partial a_{22}}{\partial c_2} = \frac{\partial L}{\partial a_{21}} \cdot b_{21} + \frac{\partial L}{\partial a_{22}} \cdot b_{22}$$

This is the element-wise product $\frac{\partial L}{\partial A} * B$, then summing along axis=1 (the axis that was broadcast):

$$\boxed{\frac{\partial L}{\partial C} = \left(\frac{\partial L}{\partial A} * B\right)\texttt{.sum(axis=1, keepdim=True)}}$$

**A note on `keepdim`:** The whole point of `keepdim` is to make the gradient's shape match the original variable's shape. The gradient of $C$ must have the same shape as $C$. When we `.sum(axis=1)` _without_ `keepdim`, PyTorch collapses that dimension entirely: a (2, 2) tensor becomes (2,) — a 1-D vector. But $C$'s original shape was (2, 1), not (2,). Using `keepdim=True` keeps the summed dimension as size 1 instead of removing it, giving us (2, 1) — exactly matching $C$.

So when do we need `keepdim=True` vs not? It depends on the original shape of the variable:

- If the original shape had a **size-1 dimension** (like (1, 64) or (32, 1)), use `keepdim=True` to preserve it
- If the original was **genuinely lower-dimensional** (like (64,) — a true 1-D tensor), don't use `keepdim` — the default sum already produces the right shape

**The rule is always the same:** whichever axis was broadcast in the forward pass, sum along that axis in the backward pass. Broadcast along axis=0 → sum along axis=0. Broadcast along axis=1 → sum along axis=1. Why? Because the broadcast axis is the axis along which the element was _copied_ — and each copy creates a separate path through the computation graph. In the backward pass, gradients flow back through _all_ of those paths, and they all converge on the single original element, so we must sum them.


### Broadcasting Through Any Element-wise Operation

So far we derived gradients for element-wise _multiplication_ with broadcasting. But the broadcasting rule — **sum backward along the axis that was broadcast** — applies to _any_ element-wise operation: addition, subtraction, division, etc. The local derivative changes depending on the operation, but the way broadcasting affects the backward pass is always the same.

Let's see this with a combined example: $X = A * B + C$, where $A$ and $B$ are 2×2, and $C$ is 1×2 (broadcast along axis=0 through the addition).

$$\begin{bmatrix} x_{11} & x_{12} \\ x_{21} & x_{22} \end{bmatrix} = \begin{bmatrix} a_{11} & a_{12} \\ a_{21} & a_{22} \end{bmatrix} * \begin{bmatrix} b_{11} & b_{12} \\ b_{21} & b_{22} \end{bmatrix} + \begin{bmatrix} c_1 & c_2 \\ c_1 & c_2 \end{bmatrix}$$

Expanding:

$$x_{11} = a_{11} \cdot b_{11} + c_1$$
$$x_{12} = a_{12} \cdot b_{12} + c_2$$
$$x_{21} = a_{21} \cdot b_{21} + c_1$$
$$x_{22} = a_{22} \cdot b_{22} + c_2$$

Given $\frac{\partial L}{\partial X}$, let's derive all three gradients.

#### Deriving $\frac{\partial L}{\partial A}$

Each $a_{ij}$ only affects $x_{ij}$, and $\frac{\partial x_{ij}}{\partial a_{ij}} = b_{ij}$:

$$\frac{\partial L}{\partial a_{11}} = \frac{\partial L}{\partial x_{11}} \cdot b_{11}, \quad \frac{\partial L}{\partial a_{12}} = \frac{\partial L}{\partial x_{12}} \cdot b_{12}, \quad \frac{\partial L}{\partial a_{21}} = \frac{\partial L}{\partial x_{21}} \cdot b_{21}, \quad \frac{\partial L}{\partial a_{22}} = \frac{\partial L}{\partial x_{22}} \cdot b_{22}$$

$$\boxed{\frac{\partial L}{\partial A} = \frac{\partial L}{\partial X} * B}$$

No broadcasting happened for $A$, so no summing — just element-wise multiply by the local derivative ($B$).

#### Deriving $\frac{\partial L}{\partial B}$

Exactly the same logic — each $b_{ij}$ only affects $x_{ij}$, and $\frac{\partial x_{ij}}{\partial b_{ij}} = a_{ij}$:

$$\frac{\partial L}{\partial b_{11}} = \frac{\partial L}{\partial x_{11}} \cdot a_{11}, \quad \frac{\partial L}{\partial b_{12}} = \frac{\partial L}{\partial x_{12}} \cdot a_{12}, \quad \frac{\partial L}{\partial b_{21}} = \frac{\partial L}{\partial x_{21}} \cdot a_{21}, \quad \frac{\partial L}{\partial b_{22}} = \frac{\partial L}{\partial x_{22}} \cdot a_{22}$$

$$\boxed{\frac{\partial L}{\partial B} = \frac{\partial L}{\partial X} * A}$$

No broadcasting for $B$ either — same shape as output, so gradient flows straight back.

#### Deriving $\frac{\partial L}{\partial C}$

Now $C$ was broadcast through the **addition** (not multiplication). The local derivative of addition is just 1 — $\frac{\partial x_{ij}}{\partial c_j} = 1$ — but $c_1$ was copied to both rows, so it influenced $x_{11}$ and $x_{21}$:

$$\frac{\partial L}{\partial c_1} = \frac{\partial L}{\partial x_{11}} \cdot \frac{\partial x_{11}}{\partial c_1} + \frac{\partial L}{\partial x_{21}} \cdot \frac{\partial x_{21}}{\partial c_1} = \frac{\partial L}{\partial x_{11}} \cdot 1 + \frac{\partial L}{\partial x_{21}} \cdot 1$$

$$\frac{\partial L}{\partial c_2} = \frac{\partial L}{\partial x_{12}} \cdot \frac{\partial x_{12}}{\partial c_2} + \frac{\partial L}{\partial x_{22}} \cdot \frac{\partial x_{22}}{\partial c_2} = \frac{\partial L}{\partial x_{12}} \cdot 1 + \frac{\partial L}{\partial x_{22}} \cdot 1$$

$$\boxed{\frac{\partial L}{\partial C} = \frac{\partial L}{\partial X}\texttt{.sum(axis=0)}}$$

Notice what happened: the local derivative for addition is just 1 (unlike multiplication where it's the other operand), so the upstream gradient passes through unchanged — but we still **sum along axis=0** because that's the axis $C$ was broadcast along. The summing comes from the broadcasting, not from the type of operation.

#### The general pattern

For _any_ element-wise operation $X = \text{op}(A, B, C, \ldots)$ where some operand is broadcast:

1. **Compute the local derivative** — this depends on the specific operation (multiplication → other operand, addition → 1, power → exponent × base^(exp-1), etc.)
2. **Multiply upstream gradient by local derivative** — element-wise, just like the scalar chain rule
3. **If the operand was broadcast, sum along the broadcast axis** — this step is purely about the shape mismatch from broadcasting, independent of what the operation was


### Derivatives Through Collapsing Operations in the Forward Pass

We saw how to handle broadcasting operations in the forward pass — a smaller tensor is copied along some axis, and in the backward pass we sum along that same axis. Now what happens if we have the _opposite_ — **collapsing operations** in the forward pass, such as summing along an axis?

This is the mirror image: if the forward pass **collapses** (sums/reduces) along an axis, the backward pass must **broadcast** the gradient back along that same axis. Collapsing many values into one in the forward pass means that single output received contributions from many inputs, so the gradient must fan back out to all of them.

#### Example 1: Sum along axis=0

Let $B$ be 2×2 and $A = B\texttt{.sum(axis=0)}$, giving $A$ shape (2,):

$$\begin{bmatrix} a_1 & a_2 \end{bmatrix} = \begin{bmatrix} b_{11} & b_{12} \\ b_{21} & b_{22} \end{bmatrix}\texttt{.sum(axis=0)}$$

Expanding:

$$a_1 = b_{11} + b_{21}$$
$$a_2 = b_{12} + b_{22}$$

Each $b_{ij}$ only affects the corresponding $a_j$ (the column it belongs to), and the local derivative of a sum is 1 for every term:

$$\frac{\partial L}{\partial b_{11}} = \frac{\partial L}{\partial a_1} \cdot \frac{\partial a_1}{\partial b_{11}} = \frac{\partial L}{\partial a_1} \cdot 1$$

$$\frac{\partial L}{\partial b_{12}} = \frac{\partial L}{\partial a_2} \cdot \frac{\partial a_2}{\partial b_{12}} = \frac{\partial L}{\partial a_2} \cdot 1$$

$$\frac{\partial L}{\partial b_{21}} = \frac{\partial L}{\partial a_1} \cdot \frac{\partial a_1}{\partial b_{21}} = \frac{\partial L}{\partial a_1} \cdot 1$$

$$\frac{\partial L}{\partial b_{22}} = \frac{\partial L}{\partial a_2} \cdot \frac{\partial a_2}{\partial b_{22}} = \frac{\partial L}{\partial a_2} \cdot 1$$

Assembling:

$$\frac{\partial L}{\partial B} = \begin{bmatrix} \frac{\partial L}{\partial a_1} & \frac{\partial L}{\partial a_2} \\ \frac{\partial L}{\partial a_1} & \frac{\partial L}{\partial a_2} \end{bmatrix}$$

This is just $\frac{\partial L}{\partial A}$ **broadcast back along axis=0** — the exact axis that was collapsed in the forward pass:

$$\boxed{\frac{\partial L}{\partial B} = \frac{\partial L}{\partial A} \text{ broadcast along axis=0 (i.e. copied to every row)}}$$

In code, this happens automatically: `dA` has shape (2,) and `dB` needs shape (2, 2). Since `dA` has no row dimension, PyTorch broadcasts it across rows when you use it in subsequent operations. If `keepdim=True` was used in the forward sum (giving `A` shape (1, 2)), then `dA` already has shape (1, 2) and broadcasting works the same way.

#### Example 2: Sum along axis=1

Now let $A = B\texttt{.sum(axis=1, keepdim=True)}$, giving $A$ shape (2, 1):

$$\begin{bmatrix} a_1 \\ a_2 \end{bmatrix} = \begin{bmatrix} b_{11} + b_{12} \\ b_{21} + b_{22} \end{bmatrix}$$

Expanding:

$$a_1 = b_{11} + b_{12}$$
$$a_2 = b_{21} + b_{22}$$

Each $b_{ij}$ only affects the corresponding $a_i$ (the row it belongs to):

$$\frac{\partial L}{\partial b_{11}} = \frac{\partial L}{\partial a_1} \cdot 1, \quad \frac{\partial L}{\partial b_{12}} = \frac{\partial L}{\partial a_1} \cdot 1, \quad \frac{\partial L}{\partial b_{21}} = \frac{\partial L}{\partial a_2} \cdot 1, \quad \frac{\partial L}{\partial b_{22}} = \frac{\partial L}{\partial a_2} \cdot 1$$

$$\frac{\partial L}{\partial B} = \begin{bmatrix} \frac{\partial L}{\partial a_1} & \frac{\partial L}{\partial a_1} \\ \frac{\partial L}{\partial a_2} & \frac{\partial L}{\partial a_2} \end{bmatrix}$$

$$\boxed{\frac{\partial L}{\partial B} = \frac{\partial L}{\partial A} \text{ broadcast along axis=1 (i.e. copied to every column)}}$$

#### Example 3: Sum with a scalar multiplier (mean)

A mean is just a sum followed by multiplication by $\frac{1}{n}$. The gradient flows backward through both operations. The $\frac{1}{n}$ scalar multiplies every element, so its local derivative is $\frac{1}{n}$. Then the sum broadcasts back along the collapsed axis. For $A = \frac{1}{n} \cdot B\texttt{.sum(axis=0, keepdim=True)}$:

$$\boxed{\frac{\partial L}{\partial B} = \frac{1}{n} \cdot \frac{\partial L}{\partial A} \text{ broadcast along axis=0}}$$

#### The symmetry

Broadcasting and collapsing are **exact mirrors** of each other:

| Forward operation               | Backward operation            |
| ------------------------------- | ----------------------------- |
| **Broadcast** (copy along axis) | **Sum** along that axis       |
| **Collapse** (sum along axis)   | **Broadcast** along that axis |

This makes intuitive sense: if the forward pass copies a value to $N$ places, the backward pass must collect $N$ gradient contributions back (sum). If the forward pass collapses $N$ values into one, the backward pass must distribute the gradient back to all $N$ contributors (broadcast).

**Case 1: Broadcast forward, sum backward** — Forward: $b_i = c$ for all $i$

<table><tr>
<td>

```mermaid
graph LR
    subgraph "FORWARD: bᵢ = c (copy)"
        c["c (1,)"] -->|copy| b1["b₁"]
        c -->|copy| b2["b₂"]
        c -->|copy| b3["b₃"]
    end
```

</td>
<td>

```mermaid
flowchart BT
    subgraph "BACKWARD: dc = Σ dbᵢ"
        direction BT
        db1["db₁"] -->|+| dc["dc (1,)"]
        db2["db₂"] -->|+| dc
        db3["db₃"] -->|+| dc
    end
```

</td>
</tr></table>

One value was copied to many places. In the backward pass, gradients from all copies flow back and **sum** into one.

**Case 2: Collapse forward, broadcast backward** — Forward: $a = \sum b_i$

<table><tr>
<td>

```mermaid
graph LR
    subgraph "FORWARD: a = Σ bᵢ (sum)"
        b1["b₁"] -->|+| a["a (1,)"]
        b2["b₂"] -->|+| a
        b3["b₃"] -->|+| a
    end
```

</td>
<td>

```mermaid
flowchart LR
    subgraph "BACKWARD: dbᵢ = da (copy)"
        direction BT
        da["da (1,)"] -->|copy| db1["db₁"]
        da -->|copy| db2["db₂"]
        da -->|copy| db3["db₃"]
    end
```

</td>
</tr></table>

Many values were collapsed into one. In the backward pass, the gradient fans back out — **broadcast** equally to all contributors.

The arrows reverse, and the operations swap: **copy** ↔ **sum**.
